# An End-to-End AI Harvest Planner for Low-Cost Fruit-Picking Robots Built on a Physics-Consistent World Model

**`01_data_preparation.ipynb`**

From detections to everything the planner reads: a pick dataset, a synthetic canopy, and a set
of settled state transitions.

Three of the four steps cost hours, so each sits behind a flag and defaults to **verifying** the
artifact it would have produced. Run top to bottom on a populated tree and it takes about a
minute, and tells you whether every input is present, well formed, and consistent with the
others. Set a flag to rebuild that step.

| Step | Flag | Cost | Produces |
|---|---|---|---|
| Pick dataset from detections | `RUN_GENERATE` | 5–7 h, 6 shards | `dataset/rows_full_v2.csv` |
| Synthetic canopy | `RUN_CANOPY` | seconds | `data/trees_resampled_stalk.csv` |
| Settled pose per fruit | `RUN_MEASURE` | 19 min | `data/trees_measured_pose.csv` |
| Transition collection | `RUN_COLLECT` | 40 min | `data/transitions/transitions.csv` |

The checks in section 5 are not decoration. Two of them would have caught errors that produced
eighty minutes of wrong numbers each: a canopy whose stalk directions were never read from the
source data, and a planning environment that fabricated the one feature the outcome model cares
most about. Both were invisible in every summary statistic until the distributions were put side
by side.


In [1]:
import os
import sys
import time
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd

# --- the only path in this notebook ---------------------------------------------------------
# Change ROOT and nothing else. Everything read and written lives under it, including the
# detections: generate.py resolves apple_crops/ and data/ from AIPICK_ROOT at import
# time, so it has to see the same root as the notebook.
# Default to the folder this notebook sits in, so a clone runs without setup. An absolute
# default only ever pointed at one machine, and an environment variable set in a shell does
# not reach a kernel that was already running.
ROOT = Path(os.environ.get("AIPICK_ROOT") or Path.cwd())
os.environ["AIPICK_ROOT"] = str(ROOT)

SRC    = ROOT/"src"
DATA   = ROOT/"data"
MODELS = ROOT/"models"
TX     = DATA/"transitions"
RUNS   = ROOT/"runs"
RUNS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

TREES_USED = "trees_measured_pose.csv"

RUN_GENERATE = False      # 5-7 h    detections -> pick dataset
RUN_CANOPY   = False      # seconds  stalk directions resampled from the detections
RUN_MEASURE  = False      # 19 min   settled pose, once per visible fruit
RUN_COLLECT  = False      # 40 min   settled state pairs across picks

pd.set_option("display.width", 200)
print(f"root {ROOT}\n")

EXPECTED = [ROOT/"apple_crops"/"manifest.csv",
            SRC/"physics.py", SRC/"generate.py", SRC/"environment.py",
            DATA/"picks.csv",
            DATA/"trees_uniform_stalk.csv", DATA/"trees_resampled_stalk.csv",
            DATA/"trees_measured_pose.csv",
            MODELS/"outcome.pkl", MODELS/"station_planner.pt",
            MODELS/"pick_policy.pt", MODELS/"dynamics.joblib",
            TX/"transitions.csv", TX/"transition_picks.csv"]
missing = [p for p in EXPECTED if not p.exists()]
for p in EXPECTED:
    mark = "ok " if p.exists() else "-- "
    size = f"{p.stat().st_size/1e6:7.1f} MB" if p.exists() else "        "
    print(f"  {mark}{size}  {p.relative_to(ROOT)}")
print(f"\n{len(EXPECTED)-len(missing)}/{len(EXPECTED)} present")
if missing:
    print("missing:", ", ".join(str(p.relative_to(ROOT)) for p in missing))




root c:\aipick\git

  ok     1.5 MB  apple_crops\manifest.csv
  ok     0.0 MB  src\physics.py
  ok     0.1 MB  src\generate.py
  ok     0.0 MB  src\environment.py
  ok    11.5 MB  data\picks.csv
  --           data\trees_uniform_stalk.csv
  --           data\trees_resampled_stalk.csv
  ok     1.2 MB  data\trees_measured_pose.csv
  ok     1.0 MB  models\outcome.pkl
  --           models\station_planner.pt
  ok     2.6 MB  models\pick_policy.pt
  ok     0.7 MB  models\dynamics.joblib
  ok     0.8 MB  data\transitions\transitions.csv
  ok     0.0 MB  data\transitions\transition_picks.csv

11/14 present
missing: data\trees_uniform_stalk.csv, data\trees_resampled_stalk.csv, models\station_planner.pt


## 1. Source detections

FRESH gives each detected fruit a position, a diameter, and a stem-direction heading. That last
one is why this dataset was chosen: its 3D box aligns its X axis with the vector from the fruit
centre to the calyx, so orientation reduces to a unit stalk vector — exactly what the physics
needs to build a scene.

The generator reads them at import time, resolving `apple_crops/` from `AIPICK_ROOT`, so the
manifest sits under the same root as everything else. It is excluded from the published
repository — see the note in the README — but a working copy keeps it here.


In [2]:
GEN = None
try:
    spec = importlib.util.spec_from_file_location("gen", SRC/"generate.py")
    GEN = importlib.util.module_from_spec(spec); GEN.__name__ = "gen"
    spec.loader.exec_module(GEN)
    print(f"generator loaded, {len(GEN.MAN):,} detections")
    print(f"  cluster sizes {GEN.CLUSTER_SIZES} with p={GEN.CLUSTER_PROBS}")
    print(f"  observation settle {GEN.OBSERVE_SETTLE} steps")
except Exception as ex:
    print(f"generator unavailable: {type(ex).__name__}: {ex}")
    print("  Needed to regenerate the dataset and for the stalk-direction check below.")
    print(f"  It reads {ROOT/'apple_crops'/'manifest.csv'}.")

if RUN_GENERATE and GEN is None:
    raise FileNotFoundError("RUN_GENERATE is on but the detections are not reachable")


generator loaded, 8,133 detections
  cluster sizes [1, 2, 3] with p=[0.2, 0.5, 0.3]
  observation settle 2400 steps


## 2. The pick dataset

One pick per fruit, evaluated at four gripper apertures **from the same settled scene**. That is
what makes the aperture comparison counterfactual rather than observational: the four rows for a
fruit differ only in the aperture, so opening the basket wider is separated from the fruit being
different.

Each scene settles for 2,400 steps before it is observed — ten seconds of simulated time. A stalk
released from its design pose swings like a pendulum and reaches equilibrium somewhere near five,
so observing earlier records the transient. This project made that mistake twice and paid for it
both times.


In [3]:
if RUN_GENERATE:
    print("Generation runs outside the notebook: it shards across processes because the MuJoCo")
    print("renderer holds a GL context that does not survive a fork.\n")
    print(f"    set AIPICK_ROOT={ROOT}")
    print(f"    cd {SRC}")
    print( "    python generate.py --launch --shards 6")
    print( "    python generate.py --merge\n")
    raise SystemExit("run those, then set RUN_GENERATE=False and rerun this cell")

DS = pd.read_csv(DATA/"picks.csv")
print(f"{len(DS):,} rows, {len(DS.columns)} columns")

if "apple_id" in DS.columns:
    per = DS.groupby("apple_id").size()
    print(f"{DS.apple_id.nunique():,} fruit, {per.mean():.2f} rows each "
          f"(min {per.min()}, max {per.max()})")
    if "aperture_ratio" in DS.columns:
        print(f"apertures {sorted(DS.aperture_ratio.unique())}, "
              f"{(per == 4).sum():,} fruit have all four")

if "split" in DS.columns:
    print(f"\nsplit {DS.split.value_counts().to_dict()}")
    if "apple_id" in DS.columns:
        leaked = (DS.groupby("apple_id").split.nunique() > 1).sum()
        print(f"  fruit in more than one split: {leaked}"
              f"   {'ok' if leaked == 0 else 'LEAK -- the four rows must move together'}")

print("\noutcome distribution (%)")
print((DS.code.value_counts(normalize=True)*100).round(2).to_string())



32,532 rows, 40 columns
8,133 fruit, 4.00 rows each (min 4, max 4)
apertures [np.float64(0.85), np.float64(1.0), np.float64(1.2), np.float64(1.45)], 8,133 fruit have all four

outcome distribution (%)
code
SUCCESS             84.58
NEIGHBOR_KNOCKED     7.55
APPROACH_BLOCKED     3.57
GRASP_FAILED         2.67
STEM_PULL            1.43
NO_DETACH            0.21
SPUR_BREAK           0.00


## 3. The synthetic canopy

The detections give clusters, not trees: a fruit and its immediate neighbours, with nothing about
how they sit on a canopy. Planning needs a whole tree, so one is synthesised — positions,
diameters, stalk directions, and visibility from three standing positions.

Three generations exist and all three are kept, because the difference between them is a result.

**`trees_uniform_stalk`** drew stalk directions from `U(0, π)` without ever reading the detections, putting the
median tilt at 44.8° against the source's 27.0°. Every layer-3 figure measured on it stood on a
canopy the outcome model had never seen the like of.

**`trees_resampled_stalk`** resamples the stalk direction and diameter from the detections' empirical distributions.
Only three columns change — positions and visibility are byte-identical — so one against the other
isolates the axis and nothing else.

**`trees_measured_pose`** adds the settled pose, measured once per fruit. The planning environment used to build
this from an interpolation table returning zero or about twenty degrees, a two-valued step
function that correlated 0.140 with what the physics settles to.


In [4]:
if RUN_CANOPY:
    if GEN is None:
        raise RuntimeError("RUN_CANOPY needs the detections")
    T1 = pd.read_csv(DATA/"trees_uniform_stalk.csv")
    SD = np.array([GEN.geom_of(r)["sdir"] for _, r in GEN.MAN.iterrows()])   # scene frame, z up
    rng = np.random.default_rng(20260815)
    draw = SD[rng.integers(0, len(SD), len(T1))]

    T2 = T1.copy()
    T2["sdir_x"], T2["sdir_y"], T2["sdir_z"] = draw[:, 0], draw[:, 2], draw[:, 1]
    assert all(T2[c].equals(T1[c]) for c in T1.columns if not c.startswith("sdir_")), \
        "the patch disturbed a column it should not have"
    dst = DATA/"trees_resampled_stalk.csv"
    assert not dst.exists(), f"{dst.name} exists -- bump the version rather than overwrite"
    T2.to_csv(dst, index=False)
    print(f"wrote {dst.name}")


def tilt_from_vertical(df):
    '''Angle between the stalk and vertical. The tree frame has y up.'''
    v = df[["sdir_x", "sdir_y", "sdir_z"]].to_numpy(float)
    v = v/np.linalg.norm(v, axis=1, keepdims=True)
    return np.degrees(np.arccos(np.clip(np.abs(v[:, 1]), 0, 1)))


rows = []
for tag, name in (("uniform stalk", "trees_uniform_stalk.csv"),
                  ("resampled stalk", "trees_resampled_stalk.csv"),
                  ("+ measured pose", "trees_measured_pose.csv")):
    p = DATA/name
    if not p.exists():
        print(f"  {tag:<22} {name} not present")
        continue
    df = pd.read_csv(p)
    t = tilt_from_vertical(df)
    rows.append(dict(canopy=tag, fruit=len(df), trees=df.tree.nunique(),
                     visible=int(df.vis_any.sum()) if "vis_any" in df else np.nan,
                     tilt_mean=t.mean(), tilt_p50=np.median(t),
                     lean_mean=df.lean_deg.mean() if "lean_deg" in df else np.nan))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\n  detections, for comparison: stalk tilt mean 32.19, median 27.00 deg")

T3 = pd.read_csv(DATA/TREES_USED)
VIS = T3[T3.vis_any] if "vis_any" in T3 else T3
if RUN_MEASURE:
    print("\nMeasuring the pose is archive/aipick_measured_pose.ipynb -- 19 minutes, resumable")
    print("by tree. Run it, then set RUN_MEASURE=False.")
    raise SystemExit("measure the pose first")

print(f"\nsettled pose, {len(VIS):,} visible fruit\n")
print(VIS.lean_deg.describe(percentiles=[.5, .9, .99]).round(3).to_string())
n = np.linalg.norm(T3[["sax_x", "sax_y", "sax_z"]].to_numpy(float), axis=1)
print(f"\n  above 1 degree: {(VIS.lean_deg > 1).mean()*100:.1f}%")
print(f"  axis unit length: max |n-1| = {np.abs(n-1).max():.2e}")


  uniform stalk          trees_uniform_stalk.csv not present
  resampled stalk        trees_resampled_stalk.csv not present
         canopy  fruit  trees  visible  tilt_mean  tilt_p50  lean_mean
+ measured pose   6000     50     4729      31.77    26.636      1.455

  detections, for comparison: stalk tilt mean 32.19, median 27.00 deg

settled pose, 4,729 visible fruit

count    4729.000
mean        1.846
std         6.408
min         0.000
50%         0.005
90%         4.145
99%        31.508
max        89.870

  above 1 degree: 12.8%
  axis unit length: max |n-1| = 3.33e-16


## 4. Transitions

What a pick does to the fruit it leaves behind, measured at equilibrium rather than at the moment
the gripper lets go.

The physics narrows the question before any model is fitted. A survivor with nothing holding it
hangs vertical: across 99 such picks the settled lean averaged 0.004° and not one exceeded five.
Only where two or three survivors remain, still touching, does the outcome vary — about twelve
per cent of those rows.

Reading the survivor the instant the pick returns gives something else: a mean of 3.76° and a
maximum of 125.9°, which is a stalk mid-swing. Re-settling brings the same fruit to 0.379° and a
maximum of 7.98°. The transient is real and nobody ever sees it, because the pick cycle is
fourteen seconds and a stalk is back to 0.3° within five.


In [5]:
if RUN_COLLECT:
    print("Collection is archive/aipick_transitions2.ipynb -- 40 minutes, resumable by tree.")
    raise SystemExit("collect the transitions first")

TXD = pd.read_csv(TX/"transitions.csv").drop_duplicates(
    ["tree", "step", "neighbour"], keep="last")
TXD["leaning"] = (TXD.post_lean_deg > 5).astype(int)

print(f"{len(TXD):,} transitions over {TXD.tree.nunique()} trees\n")
agg = dict(n=("leaning", "size"), leaning=("leaning", "mean"),
           pre=("pre_lean_deg", "mean"), settled=("post_lean_deg", "mean"),
           settled_max=("post_lean_deg", "max"))
if "now_lean_deg" in TXD.columns:
    agg["transient"] = ("now_lean_deg", "mean")
print(TXD.groupby("n_surv").agg(**agg).round(4).to_string())

print("\n  n_surv 1 is decided by the physics; 2 and 3 are the rows a model is for.")
if "now_lean_deg" in TXD.columns:
    print(f"  transient vs settled: mean {TXD.now_lean_deg.mean():.2f} -> "
          f"{TXD.post_lean_deg.mean():.2f} deg, "
          f"max {TXD.now_lean_deg.max():.1f} -> {TXD.post_lean_deg.max():.1f}")


2,630 transitions over 48 trees

           n  leaning     pre  settled  settled_max  transient
n_surv                                                        
1         99   0.0000  1.1695   0.0042       0.1524     2.6455
2        416   0.1346  2.7971   2.6504      73.8496     3.6537
3       2115   0.1173  3.4903   2.4232      64.7624     5.2664

  n_surv 1 is decided by the physics; 2 and 3 are the rows a model is for.
  transient vs settled: mean 4.91 -> 2.37 deg, max 143.3 -> 73.8


## 5. Distribution checks

Three comparisons, each of which exists because something slipped past every summary statistic
until the two sides were put next to each other.

**Canopy against detections.** Whether the synthetic stalk directions look like the measured
ones. The uniform draw fails by a wide margin, and that failure is why the resampled canopy exists.

**Canopy pose against the dataset.** The outcome model was fitted on poses the generator
measured. If the canopy's settled poses are shaped differently, the model extrapolates every time
the planner calls it — which is what produced a predicted success rate of 0.980 against a
realised 0.636.

**Planning environment against the physics.** The last link: what `TreeGeometry` reports for a
fruit, against what MuJoCo settles it to. The old interpolation table scored 0.140 here.


In [6]:
def summarise(name, v, q=(50, 75, 90, 99)):
    v = np.asarray(v, float)
    return dict(source=name, n=len(v), mean=v.mean(), above_1_deg=(v > 1).mean()*100,
                **{f"p{x}": np.percentile(v, x) for x in q})


rows = []
if GEN is not None:
    sd = np.array([GEN.geom_of(r)["sdir"] for _, r in GEN.MAN.iterrows()])
    rows.append(summarise("detections", np.degrees(np.arccos(np.clip(np.abs(sd[:, 2]), 0, 1)))))
else:
    rows.append(dict(source="detections (published)", n=8133, mean=32.19,
                     above_1_deg=np.nan, p50=27.00, p75=np.nan, p90=np.nan, p99=np.nan))
for tag, name in (("uniform stalk", "trees_uniform_stalk.csv"),
                  ("resampled stalk", "trees_resampled_stalk.csv"),
                  ("measured pose", "trees_measured_pose.csv")):
    p = DATA/name
    if p.exists():
        rows.append(summarise(f"canopy: {tag}", tilt_from_vertical(pd.read_csv(p))))

print("check 1 -- stalk direction\n")
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\n  the uniform draw sits near 45 because U(0, pi) makes the angle from vertical U(0, 90).")

print("\n\ncheck 2 -- settled pose against the dataset the outcome model was fitted on\n")
ds_lean = (DS.drop_duplicates("apple_id") if "apple_id" in DS else DS).obs_lean_deg
print(pd.DataFrame([summarise("dataset (measured)", ds_lean),
                    summarise("canopy (measured)", VIS.lean_deg)]).round(3).to_string(
    index=False))
gap = abs(VIS.lean_deg.mean() - ds_lean.mean())
print(f"\n  distance between the means {gap:.3f} deg   "
      f"{'PASS' if gap < 2.0 else 'FAIL -- do not build on this'}")

print("\n\ncheck 3 -- what the planner reports, against what the physics settles to\n")
try:
    import environment as E3
    E3.load(ROOT, trees=TREES_USED, dynamics=False)
    E3._CACHE.clear()
    pairs = []
    for tid, g_ in TXD.groupby("tree"):
        gg, _, _ = E3.tree_cache(int(tid), 0.200, 0.600)
        idx = {int(f): i for i, f in enumerate(gg.ids)}
        obs = gg.observe_all(np.ones(gg.n, bool))
        for _, r in g_.iterrows():
            i = idx.get(int(r.neighbour))
            if i is not None:
                pairs.append((float(obs[i, gg.COL["obs_lean_deg"]]), float(r.pre_lean_deg)))
    GT = pd.DataFrame(pairs, columns=["env", "mujoco"])
    corr = GT.corr().iloc[0, 1]
    print(f"  {len(GT):,} fruit   env mean {GT.env.mean():.3f}   "
          f"mujoco mean {GT.mujoco.mean():.3f}")
    print(f"  correlation {corr:+.3f}   MAE {(GT.env - GT.mujoco).abs().mean():.3f} deg")
    print(f"  the interpolation table scored +0.140 with an MAE of 7.378")
    print(f"  {'PASS' if corr >= 0.6 else 'FAIL'}")
except Exception as ex:
    print(f"  skipped: {type(ex).__name__}: {ex}")


check 1 -- stalk direction

               source    n  mean  above_1_deg   p50   p75   p90  p99
           detections 8133 32.19        97.27 27.00 45.22 70.67 90.0
canopy: measured pose 6000 31.77        96.97 26.64 44.12 70.00 90.0

  the uniform draw sits near 45 because U(0, pi) makes the angle from vertical U(0, 90).


check 2 -- settled pose against the dataset the outcome model was fitted on

            source    n  mean  above_1_deg   p50   p75   p90    p99
dataset (measured) 8133 2.409        9.873 0.002 0.084 0.933 43.380
 canopy (measured) 4729 1.846       12.815 0.005 0.149 4.145 31.508

  distance between the means 0.562 deg   PASS


check 3 -- what the planner reports, against what the physics settles to

  2,630 fruit   env mean 2.974   mujoco mean 3.293
  correlation +0.705   MAE 1.841 deg
  the interpolation table scored +0.140 with an MAE of 7.378
  PASS


### What comes next

`02_integrated_model` trains the four learned components on these files and assembles the chain,
in the order the problem builds up: what a pick does to its neighbours, how to order a cluster,
and how to plan a tree.
